# 🎓 Semester at the study office · starter notebook

**The situation.** About one first-year student in five leaves the university during the first semester. Many of
them could have been helped: a conversation with an adviser, a study group, help with money. But the study office
has only **three advisers**. At the end of **week 6** they can hold about **40 conversations**.

**Your job:** build a model that helps the office decide *who* to talk to, find out what **mistakes** it makes
(students worried for nothing, students missed), turn it into a **rule** the office can use, and then build a small
**Streamlit app** for the office.

**How to work.** Go through the steps in order. Each ✏️ cell is yours to write. Use the hotel notebook (session 10) as your template: it
does every step for hotel bookings. AI assistants are welcome for the code, but make sure you understand each step:
you will explain it in your video. `GUIDE.md` explains the terms on one page.

*The students are synthetic, but every profile is resampled from a real student (UCI "Predict students' dropout
and academic success", 4,424 students, CC BY 4.0), and the risk of leaving follows a model fitted on the real
outcomes.*

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import xgboost as xgb

# the data lives on GitHub, so this works in Colab without uploading anything
URL = "https://raw.githubusercontent.com/aaubs/ds-master/main/assignments/study-office/data/"
BASE = "data/" if os.path.exists("data/history_week6.csv") else URL
history = pd.read_csv(BASE + "history_week6.csv")   # past students, with what happened
new = pd.read_csv(BASE + "new_week6.csv")           # this year's students, week 6: what happens is not known yet
print(history.shape, new.shape)
history.head()

---
## 0 · Think first 🤔

Before any code, write down in your own words:

1. **Who decides what?** Who uses the model's output, and what exactly do they decide?
2. **When?** What does the office know about a student at the end of week 6, and what does it not know yet?
3. **The two mistakes.** The model can flag a student who would have stayed anyway (a **false alarm**, a *false
   positive*), or miss a student who then leaves (a **missed student**, a *false negative*). What does each cost,
   and for whom?
4. **Which is worse?** Is one false alarm as bad as one missed student?

*Your answers:*

1. 
2. 
3. 
4. 

---
## 1 · Look at the data

**`history_week6.csv`**: one row per student from the 2023, 2024 and 2025 cohorts who was still enrolled at the end
of week 6, with what the office knew then, and **`left`** (1 = left later that semester). **`new_week6.csv`**: the
2026 students, this week. This is the list the office has to act on.

| Column | Meaning |
|---|---|
| `age`, `gender`, `programme`, `evening_programme` | from enrolment |
| `admission_grade` | grade point average from upper secondary school (Danish scale) |
| `international`, `first_gen` | international student; first in the family at university |
| `su_scholarship`, `fees_owed`, `moved_from_home`, `married` | money and life situation (0/1) |
| `logins_total`, `logins_last3`, `logins_trend` | logins on the learning platform: weeks 1–6, the last three weeks, and the change |
| `submitted_share`, `missed_last3` | share of weekly assignments handed in; assignments missed in the last three weeks |
| `quiz_mean`, `weeks_since_login` | average quiz score; weeks since the last login |
| `ects_passed_sem1`, `deregistration_form_opened`, `last_login_week` | ⚠️ see step 2 |
| `left` | **the target**: 1 = left later in the semester |

✏️ **Your turn.** How many students leave? Compare the share who left across a few columns you think matter, for example `programme`, `fees_owed`, `submitted_share`. (Template: section 1 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

`history['left'].mean()`; `history.groupby('fees_owed')['left'].mean()`; for a number, cut it into groups first: `history.groupby(pd.cut(history['submitted_share'], [0, .5, .8, 1], include_lowest=True))['left'].mean()`.
</details>

In [ ]:
# your code here

---
## 2 · What is known when? (leakage)

The model will be used **at the end of week 6**. A column that is only filled in later makes the model look
brilliant here and useless in October. Compare the two files: which columns does the history have that this week's
list does not?

In [ ]:
sorted(set(history.columns) - set(new.columns))

✏️ **Your turn.** For each of these columns: when is it recorded? Would the office know it at the end of week 6? Write one sentence each, then make a list `features` of the columns the model may use (not `student_id`, `cohort`, the leaks, or the target). (Template: the leakage hunt, section 3 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

ECTS points are awarded after the exams. A deregistration form is opened by students who are about to leave. The *last* login week is computed over the whole semester. `left` is the target itself.
</details>

In [ ]:
# your code here

*Your answer:* 

---
## 3 · A split by time

The model will be used on **next year's** students, so the split follows time: learn from the 2023 and 2024
cohorts, check on 2025.

✏️ **Your turn.** Make `train` (2023 and 2024) and `val` (2025). How many students are in each, and what share leaves? (Template: section 2 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

`train = history[history['cohort'] <= 2024]` and `val = history[history['cohort'] == 2025].copy()`.
</details>

In [ ]:
# your code here

---
## 4 · Train a model

Train two models on `train` and compare them on `val` with the **AUC**: pick one student who left and one who
stayed; the AUC is how often the model gives the one who left the higher risk (0.5 = coin flip, 1 = perfect).

1. A **logistic regression**: easy to read. Which features push the risk up or down?
2. **XGBoost**, as in the hotel notebook (session 10). It may not win here, and that is a result, not a failure.

Build both as a pipeline: first prepare the data (fill gaps, scale numbers, one-hot the categories), then the
model. Use the same preparation as the hotel notebook, so you can store the model the same way in step 9.

✏️ **Your turn.** Train the logistic regression, compute its validation AUC, and plot its coefficients.

<details><summary>Hint</summary>

```python
numeric = [c for c in features if c not in ["programme", "gender"]]
categorical = ["programme", "gender"]
prepare = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False), categorical)])
logit = make_pipeline(prepare, LogisticRegression(max_iter=2000))
logit.fit(train[features], train["left"])
val["risk"] = logit.predict_proba(val[features])[:, 1]
```
Coefficients: `logit[-1].coef_[0]`, with names from `logit[0].get_feature_names_out()`.
</details>

In [ ]:
# your code here

*Your answer:* 

✏️ **Your turn.** Train XGBoost the same way and compare the AUC. Pick one model and store its predictions for 2025 in `val['risk']`: all the next steps use it.

<details><summary>Hint</summary>

`make_pipeline(prepare, xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3))`. If you want the app to be as easy as the hotel app, use XGBoost there: the hotel's way of storing the model is written for XGBoost.
</details>

In [ ]:
# your code here

---
## 5 · The mistakes: the confusion matrix ⚠️

**This is the core of the assignment.** Suppose the office talks to every student with a risk of at least 50 %.
Every student lands in one of four boxes:

| | **not contacted** | **contacted** |
|---|---|---|
| **stayed** | ✅ true negative (TN) | 📞 **false positive (FP)**: a false alarm, a worried student who was fine |
| **left** | 🚪 **false negative (FN)**: a missed student, nobody tried | 🎯 true positive (TP): reached in time |

| Question | Name | Formula |
|---|---|---|
| Of the students we contacted, how many were really at risk? | **precision** | TP / (TP + FP) |
| Of the students who left, how many did we reach? | **recall** | TP / (TP + FN) |
| Of the students who were fine, how many did we worry? | **false positive rate** | FP / (FP + TN) |
| How many students did we handle correctly? | **accuracy** | (TP + TN) / all |

✏️ **Your turn.** Count the four boxes for a 50 % cut-off, compute the four numbers, and write each as a sentence about students (*"of the students we contacted, … were really at risk"*). Then compute the accuracy of contacting **nobody**. What does it tell you about accuracy? (Template: *Reading the confusion matrix*, section 7 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

`TN, FP, FN, TP = confusion_matrix(val['left'], val['risk'] >= 0.5).ravel()`. Contacting nobody gets every student who stayed right, so its accuracy is `1 - val['left'].mean()`.
</details>

In [ ]:
# your code here

*Your answer:* 

✏️ **Your turn.** Now move the cut-off: make a small table for cut-offs 0.05, 0.1, 0.2, 0.3, 0.5 with the number contacted, the four boxes, precision and recall. What happens to the missed students and the false alarms as the cut-off goes down?

<details><summary>Hint</summary>

Write a function `boxes(cut)` that returns a dict with the counts and the rates, then `pd.DataFrame([boxes(c) for c in [0.05, 0.1, 0.2, 0.3, 0.5]])`.
</details>

In [ ]:
# your code here

*Your answer:* 

---
## 6 · A cut-off from costs 💶

Which cut-off is right depends on what each mistake costs. These are **assumptions**; part of your job is to
question them.

- **A conversation** costs about **500 DKK** of adviser time.
- **A false alarm** costs a worried student. Say that is worth **2,000 DKK**.
- **A student who leaves** costs the university about **60,000 DKK** (and the student far more). A conversation
  does not save everyone: say it keeps **30 %** of the students at risk.

So contacting a student with risk *p* is worth on average **p × 0.30 × 60,000 − 500 − (1 − p) × 2,000**.

In [ ]:
COST_TALK, COST_WORRY, COST_LEAVE, HELPS = 500, 2_000, 60_000, 0.30

✏️ **Your turn.** Compute the net value of the rule for cut-offs from 0.02 to 0.9, plot it, and find the best cut-off. How many students would the office contact? Then change one assumption (a false alarm worth 10,000 DKK, or conversations that help only 10 %) and see how the rule moves. Who should decide these numbers? (Template: *A threshold from costs*, section 7 of the hotel notebook (session 10).)

<details><summary>Hint</summary>

Net value at a cut-off = TP × HELPS × COST_LEAVE − (TP + FP) × COST_TALK − FP × COST_WORRY. Break-even: the formula above is zero at p = (500 + 2,000) / (0.30 × 60,000 + 2,000).
</details>

In [ ]:
# your code here

*Your answer:* 

---
## 7 · Only 40 conversations

The office can talk to **40** students at week 6. With a fixed capacity the rule becomes: **talk to the 40 with the
highest risk**. How many of the 40 were really at risk (**precision at 40**)? How many of all students who left were
among them (**recall at 40**)?

✏️ **Your turn.** Compute precision and recall at 40. How many conversations would you need to reach half of the students who leave? What would a fourth adviser (about 13 more conversations) buy?

<details><summary>Hint</summary>

`top = val.nlargest(40, 'risk')`; precision = `top['left'].mean()`; recall = `top['left'].sum() / val['left'].sum()`. Try other numbers instead of 40.
</details>

In [ ]:
# your code here

*Your answer:* 

---
## 8 · Fair to whom? ⚖️

About a fifth of the students are international. Compare the mistakes **per group** for the top-40 rule.

✏️ **Your turn.** For international and domestic students: the share who really leave, the average predicted risk, and the recall of the top-40 rule. Does the model get both groups right? Then compare how often each group logs in. Is a login the same signal for everyone? What would you tell the university?

<details><summary>Hint</summary>

Mark the top 40 first: `val['top40'] = val['risk'].rank(ascending=False) <= 40`. Then `val.groupby('international')[['left', 'risk']].mean()`; recall per group = among the students who left, the share with `top40`: `val[val['left'] == 1].groupby('international')['top40'].mean()`. Logins: `history.groupby('international')['logins_total'].mean()`.
</details>

In [ ]:
# your code here

*Your answer:* 

---
## 9 · Store the model

The app needs your model as files. Do it the way the hotel notebook (session 10) does in section 9, *Storing the model properly*: the
**portable copy** (`preprocess.json` + `booster.json`) loads in the app with any Python version, which a pickle does
not. The hotel app's `portable.py` does both the saving and the loading, for a pipeline built like the one in step 4.

✏️ **Your turn.** Save your XGBoost pipeline into a folder `model`, load it back with `portable.Model`, and check that it gives the same predictions. Download the folder as a zip. You will put it into your app's repository.

<details><summary>Hint</summary>

```python
import urllib.request, shutil
urllib.request.urlretrieve("https://raw.githubusercontent.com/aaubs/tonights-front-desk/main/portable.py", "portable.py")
import portable
os.makedirs("model", exist_ok=True)
portable.export(booster, "model")
back = portable.Model("model").predict_proba(val)
print(np.allclose(back, booster.predict_proba(val[features])[:, 1], atol=1e-5))
shutil.make_archive("model", "zip", "model")   # then download model.zip (Colab: the folder icon on the left)
```
</details>

In [ ]:
# your code here

---
## 10 · This week's list

Now use the model the way the office would: on the 2026 students at the end of week 6.

✏️ **Your turn.** Predict the risk for every student in `new`, and show the 40 students the office should talk to this week. Would you send all 40 the same message? Is there anyone you would *not* contact?

<details><summary>Hint</summary>

`new['risk'] = booster.predict_proba(new[features])[:, 1]`, then `new.nlargest(40, 'risk')`.
</details>

In [ ]:
# your code here

*Your answer:* 

---
## 11 · Build the app 🛠️

Now build a Streamlit app for someone at the study office who is **not** a data scientist. Use the hotel app as your
template: [github.com/aaubs/tonights-front-desk](https://github.com/aaubs/tonights-front-desk) (`app.py`, `portable.py`,
`requirements.txt`, `model/`).

Your app should at least show:

1. **This week's list**: the 2026 students, ranked by risk, with the top 40 marked.
2. **The mistakes of a rule**: a slider (cut-off, or number of conversations) and, on the 2025 cohort, the four boxes
   in plain words ("12 students reached in time, 28 worried for nothing, 30 missed"), precision and recall.
3. **Per group**: the same numbers for international and domestic students.
4. **One thing of your own** that would help the office decide (for example the costs from step 6 as inputs, or why
   a student is on the list).

The app can read the data straight from GitHub, like this notebook (`URL + "new_week6.csv"`).

**Deploy:** create a public GitHub repository with `app.py`, `portable.py`, `requirements.txt` (`streamlit`,
`pandas`, `xgboost-cpu`, no version numbers) and your `model/` folder; then on
[share.streamlit.io](https://share.streamlit.io): *Create app* → your repository → `app.py`.